In [ ]:
import argparse
import csv
import math
import os
import random
import sys
from dataclasses import dataclass
from typing import Dict, List, Tuple

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset, Subset


# ============================================================
# Safe torchvision import
# ============================================================

def safe_import_torchvision():
    try:
        from torchvision import datasets, transforms, models
        return datasets, transforms, models
    except Exception:
        for name in list(sys.modules.keys()):
            if name.startswith("torchvision"):
                del sys.modules[name]

        from torch.library import Library
        lib = Library("torchvision", "DEF")
        try:
            lib.define("nms(Tensor dets, Tensor scores, float iou_threshold) -> Tensor")
        except Exception:
            pass

        from torchvision import datasets, transforms, models
        return datasets, transforms, models


datasets, transforms, models = safe_import_torchvision()


# ============================================================
# Utils
# ============================================================

def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    if hasattr(torch.backends, 'cudnn'):
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False


def clone_state_dict(state_dict: Dict[str, torch.Tensor]) -> Dict[str, torch.Tensor]:
    return {k: v.detach().cpu().clone() for k, v in state_dict.items()}


def load_state_dict_to_module(module: nn.Module, state_dict: Dict[str, torch.Tensor], device: str) -> None:
    module.load_state_dict({k: v.to(device) for k, v in state_dict.items()})


def subtract_state_dict(
    a: Dict[str, torch.Tensor],
    b: Dict[str, torch.Tensor],
) -> Dict[str, torch.Tensor]:
    return {k: a[k].float() - b[k].float() for k in a.keys()}


def flatten_state_dict(state_dict: Dict[str, torch.Tensor]) -> torch.Tensor:
    parts = []
    for v in state_dict.values():
        parts.append(v.reshape(-1).float().cpu())
    if not parts:
        return torch.zeros(1)
    return torch.cat(parts)


def count_trainable_params(module: nn.Module) -> int:
    return sum(p.numel() for p in module.parameters() if p.requires_grad)


# ============================================================
# DCT utilities
# ============================================================

def dct_1d_torch(x: torch.Tensor) -> torch.Tensor:
    x = x.flatten().float().cpu()
    n = x.shape[0]
    if n == 0:
        return x
    if n == 1:
        return x.clone()

    v = torch.cat([x[::2], x[1::2].flip(0)])
    Vc = torch.fft.fft(v)

    k = torch.arange(n, dtype=torch.float32)
    factor = -math.pi * k / (2.0 * n)
    W_r = torch.cos(factor)
    W_i = torch.sin(factor)

    out = 2.0 * (Vc[:n].real * W_r - Vc[:n].imag * W_i)
    return out


def dct_2d_torch(x: torch.Tensor) -> torch.Tensor:
    x = x.float().cpu()
    if x.ndim != 2:
        raise ValueError("dct_2d_torch expects a 2D tensor.")

    rows = torch.stack([dct_1d_torch(row) for row in x], dim=0)
    cols = torch.stack([dct_1d_torch(col) for col in rows.transpose(0, 1)], dim=0)
    return cols.transpose(0, 1).contiguous()


def lowfreq_block_2d(
    coeff: torch.Tensor,
    keep_ratio: float,
    min_keep: int,
    max_keep: int,
) -> torch.Tensor:
    h, w = coeff.shape
    kh = max(min_keep, int(h * keep_ratio))
    kw = max(min_keep, int(w * keep_ratio))
    kh = min(kh, max_keep, h)
    kw = min(kw, max_keep, w)
    return coeff[:kh, :kw].reshape(-1)


def low_frequency_signature(
    delta_state: Dict[str, torch.Tensor],
    keep_ratio: float = 0.02,
    min_keep_per_tensor: int = 2,
    max_keep_per_tensor: int = 8,
) -> torch.Tensor:
    parts = []

    for _, v in delta_state.items():
        x = v.detach().float().cpu()

        if x.ndim == 4:
            per_kernel_parts = []
            for o in range(x.shape[0]):
                for i in range(x.shape[1]):
                    coeff = dct_2d_torch(x[o, i])
                    per_kernel_parts.append(
                        lowfreq_block_2d(
                            coeff,
                            keep_ratio=keep_ratio,
                            min_keep=min_keep_per_tensor,
                            max_keep=max_keep_per_tensor,
                        )
                    )
            if per_kernel_parts:
                parts.append(torch.cat(per_kernel_parts))

        elif x.ndim == 2:
            coeff = dct_2d_torch(x)
            parts.append(
                lowfreq_block_2d(
                    coeff,
                    keep_ratio=keep_ratio,
                    min_keep=min_keep_per_tensor,
                    max_keep=max_keep_per_tensor,
                )
            )

        elif x.ndim == 1:
            coeff = dct_1d_torch(x)
            keep = max(min_keep_per_tensor, int(coeff.numel() * keep_ratio))
            keep = min(keep, max_keep_per_tensor, coeff.numel())
            if keep > 0:
                parts.append(coeff[:keep])

        else:
            flat = x.reshape(-1)
            coeff = dct_1d_torch(flat)
            keep = max(min_keep_per_tensor, int(coeff.numel() * keep_ratio))
            keep = min(keep, max_keep_per_tensor, coeff.numel())
            if keep > 0:
                parts.append(coeff[:keep])

    if not parts:
        return torch.zeros(1)
    return torch.cat(parts)


# ============================================================
# Dynamic / rotation utilities
# ============================================================

def rotational_feature(delta_vec: torch.Tensor) -> torch.Tensor:
    return torch.atan(delta_vec.float().cpu())


def build_rotation_scores(delta_vecs_in_order: List[torch.Tensor]) -> List[float]:
    if len(delta_vecs_in_order) == 0:
        return []

    theta = [rotational_feature(d) for d in delta_vecs_in_order]

    omega = []
    for i in range(len(theta)):
        if i == 0:
            omega.append(theta[i] / (2.0 * math.pi))
        else:
            omega.append((theta[i] - theta[i - 1]) / (2.0 * math.pi))

    n = len(omega)
    maj = n // 2 + 1
    scores = []

    for i in range(n):
        distances = []
        for j in range(n):
            if i == j:
                continue
            d = torch.norm(omega[i] - omega[j], p=2).item()
            distances.append(float(d))
        distances = sorted(distances)
        scores.append(float(sum(distances[:min(maj, len(distances))])))

    return scores


# ============================================================
# Config
# ============================================================

@dataclass
class Config:
    data_root: str = "./data"
    batch_size: int = 128
    num_workers: int = 2
    rounds: int = 50
    clients: int = 10
    malicious_clients: int = 2
    malicious_client_ids: str = "8,9"
    local_epochs: int = 1
    malicious_local_epochs: int = 2
    lr: float = 0.001
    momentum: float = 0.9
    weight_decay: float = 1e-4
    seed: int = 0
    iid_rate: float = 0.8
    poison_fraction: float = 0.75
    train_augmentation: bool = True
    target_label: int = 2
    trigger_size: int = 4
    trigger_value: float = 1.0
    defense: bool = True
    use_rotation: bool = True
    low_freq_keep_ratio: float = 0.02
    min_keep_per_tensor: int = 2
    max_keep_per_tensor: int = 8
    device: str = "cuda" if torch.cuda.is_available() else "cpu"
    save_dir: str = "./outputs_paper_aligned"


# ============================================================
# Trigger / poisoned dataset
# ============================================================

def add_pixel_trigger(images: torch.Tensor, trigger_size: int, value: float) -> torch.Tensor:
    x = images.clone()
    # CIFAR-10 tensors are normalized before the trigger is applied.
    # So we write the *normalized* equivalent of a bright red patch:
    # raw RGB = (value, 0, 0) -> normalized channel values.
    cifar10_mean = (0.4914, 0.4822, 0.4465)
    cifar10_std = (0.2470, 0.2435, 0.2616)

    r = (value - cifar10_mean[0]) / cifar10_std[0]
    g = (0.0 - cifar10_mean[1]) / cifar10_std[1]
    b = (0.0 - cifar10_mean[2]) / cifar10_std[2]

    x[:, 0, -trigger_size:, -trigger_size:] = r
    if x.shape[1] > 1:
        x[:, 1, -trigger_size:, -trigger_size:] = g
    if x.shape[1] > 2:
        x[:, 2, -trigger_size:, -trigger_size:] = b
    return x


class PoisonedSubset(Dataset):
    def __init__(
        self,
        base_dataset: Dataset,
        indices: List[int],
        poison_fraction: float,
        target_label: int,
        trigger_size: int,
        trigger_value: float,
        train: bool = True,
        seed: int = 0,
    ) -> None:
        self.base_dataset = base_dataset
        self.indices = list(indices)
        self.poison_fraction = poison_fraction
        self.target_label = target_label
        self.trigger_size = trigger_size
        self.trigger_value = trigger_value
        self.train = train

        rng = random.Random(seed)
        eligible_positions = []
        for pos, ds_idx in enumerate(self.indices):
            y = int(getattr(self.base_dataset, 'targets')[ds_idx])
            if y != self.target_label:
                eligible_positions.append(pos)

        if train:
            n_poison = int(len(eligible_positions) * poison_fraction)
            rng.shuffle(eligible_positions)
            self.poison_positions = set(eligible_positions[:n_poison])
        else:
            self.poison_positions = set(eligible_positions)

    def __len__(self) -> int:
        return len(self.indices)

    def __getitem__(self, idx: int):
        x, y = self.base_dataset[self.indices[idx]]
        if idx in self.poison_positions:
            x = add_pixel_trigger(x.unsqueeze(0), self.trigger_size, self.trigger_value).squeeze(0)
            y = self.target_label
        return x, y


# ============================================================
# Partition
# ============================================================

def partition_main_label_non_iid(
    targets: List[int],
    num_clients: int,
    iid_rate: float,
    seed: int,
) -> Dict[int, List[int]]:
    rng = random.Random(seed)
    num_classes = len(set(int(y) for y in targets))
    all_indices = list(range(len(targets)))
    rng.shuffle(all_indices)

    total_per_client = len(targets) // num_clients
    iid_per_client = int(total_per_client * iid_rate)
    non_iid_per_client = total_per_client - iid_per_client

    client_to_indices: Dict[int, List[int]] = {i: [] for i in range(num_clients)}

    # 1) IID part without replacement
    ptr = 0
    used = set()
    for cid in range(num_clients):
        chosen = all_indices[ptr:ptr + iid_per_client]
        client_to_indices[cid].extend(chosen)
        used.update(chosen)
        ptr += iid_per_client

    # 2) Remaining samples are grouped by label
    label_to_indices: Dict[int, List[int]] = {c: [] for c in range(num_classes)}
    for idx, y in enumerate(targets):
        if idx not in used:
            label_to_indices[int(y)].append(idx)

    for lst in label_to_indices.values():
        rng.shuffle(lst)

    # 3) Assign one main label per client
    main_labels = [i % num_classes for i in range(num_clients)]
    rng.shuffle(main_labels)

    for cid in range(num_clients):
        label = main_labels[cid]
        take = min(non_iid_per_client, len(label_to_indices[label]))
        chosen = label_to_indices[label][:take]
        label_to_indices[label] = label_to_indices[label][take:]
        client_to_indices[cid].extend(chosen)

    # 4) Fill any shortages from leftovers without replacement
    leftovers = []
    for lst in label_to_indices.values():
        leftovers.extend(lst)
    rng.shuffle(leftovers)

    p = 0
    for cid in range(num_clients):
        while len(client_to_indices[cid]) < total_per_client and p < len(leftovers):
            client_to_indices[cid].append(leftovers[p])
            p += 1

    return client_to_indices




def parse_malicious_client_ids(cfg: Config) -> List[int]:
    if cfg.malicious_client_ids.strip():
        ids = [int(x.strip()) for x in cfg.malicious_client_ids.split(',') if x.strip()]
        ids = sorted(set(i for i in ids if 0 <= i < cfg.clients))
        if ids:
            return ids
    # Fallback: use the last K clients so the backdoor is not immediately washed out by many benign clients
    start = max(0, cfg.clients - cfg.malicious_clients)
    return list(range(start, cfg.clients))

# ============================================================
# ResNet18 split aligned closer to paper Table I
# head params ~= 9536, tail params ~= 5130 for CIFAR-10
# ============================================================

class FlattenTail(nn.Module):
    def __init__(self, fc: nn.Module):
        super().__init__()
        self.fc = fc

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = torch.flatten(x, 1)
        x = self.fc(x)
        return x


def build_split_resnet18(num_classes: int = 10) -> Tuple[nn.Module, nn.Module, nn.Module]:
    model = models.resnet18(weights=None, num_classes=num_classes)

    head = nn.Sequential(
        model.conv1,
        model.bn1,
        model.relu,
        model.maxpool,
    )

    backbone = nn.Sequential(
        model.layer1,
        model.layer2,
        model.layer3,
        model.layer4,
        model.avgpool,
    )

    tail = FlattenTail(model.fc)
    return head, backbone, tail


# ============================================================
# Data
# ============================================================

def build_dataloaders(cfg: Config):
    cifar10_mean = (0.4914, 0.4822, 0.4465)
    cifar10_std = (0.2470, 0.2435, 0.2616)

    if cfg.train_augmentation:
        transform_train = transforms.Compose([
            transforms.RandomCrop(32, padding=4),
            transforms.RandomHorizontalFlip(),
            transforms.ToTensor(),
            transforms.Normalize(cifar10_mean, cifar10_std),
        ])
    else:
        transform_train = transforms.Compose([
            transforms.ToTensor(),
            transforms.Normalize(cifar10_mean, cifar10_std),
        ])
    transform_test = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize(cifar10_mean, cifar10_std),
    ])

    train_set = datasets.CIFAR10(
        root=cfg.data_root,
        train=True,
        download=True,
        transform=transform_train,
    )
    test_set = datasets.CIFAR10(
        root=cfg.data_root,
        train=False,
        download=True,
        transform=transform_test,
    )

    partitions = partition_main_label_non_iid(train_set.targets, cfg.clients, cfg.iid_rate, cfg.seed)
    malicious_ids = set(parse_malicious_client_ids(cfg))

    client_loaders = {}
    for cid, idxs in partitions.items():
        if cid in malicious_ids:
            ds = PoisonedSubset(
                train_set,
                idxs,
                poison_fraction=cfg.poison_fraction,
                target_label=cfg.target_label,
                trigger_size=cfg.trigger_size,
                trigger_value=cfg.trigger_value,
                train=True,
                seed=cfg.seed + cid,
            )
        else:
            ds = Subset(train_set, idxs)

        client_loaders[cid] = DataLoader(
            ds,
            batch_size=cfg.batch_size,
            shuffle=True,
            num_workers=cfg.num_workers,
            pin_memory=torch.cuda.is_available(),
        )

    clean_test_loader = DataLoader(
        test_set,
        batch_size=cfg.batch_size,
        shuffle=False,
        num_workers=cfg.num_workers,
        pin_memory=torch.cuda.is_available(),
    )

    return client_loaders, clean_test_loader


# ============================================================
# Train / eval
# ============================================================

def make_optimizer(module: nn.Module, cfg: Config) -> torch.optim.Optimizer:
    return torch.optim.SGD(
        module.parameters(),
        lr=cfg.lr,
        momentum=cfg.momentum,
        weight_decay=cfg.weight_decay,
    )


def train_one_client(
    head: nn.Module,
    backbone: nn.Module,
    tail: nn.Module,
    loader: DataLoader,
    cfg: Config,
    local_epochs: int,
) -> Tuple[Dict[str, torch.Tensor], Dict[str, torch.Tensor], Dict[str, torch.Tensor], float]:
    head.train()
    backbone.train()
    tail.train()

    head_opt = make_optimizer(head, cfg)
    back_opt = make_optimizer(backbone, cfg)
    tail_opt = make_optimizer(tail, cfg)

    total_loss = 0.0
    steps = 0

    for _ in range(local_epochs):
        for x, y in loader:
            x = x.to(cfg.device)
            y = y.to(cfg.device)

            head_opt.zero_grad()
            back_opt.zero_grad()
            tail_opt.zero_grad()

            smashed = head(x)
            features = backbone(smashed)
            logits = tail(features)

            loss = F.cross_entropy(logits, y)
            loss.backward()

            head_opt.step()
            back_opt.step()
            tail_opt.step()

            total_loss += float(loss.item())
            steps += 1

    return (
        clone_state_dict(head.state_dict()),
        clone_state_dict(backbone.state_dict()),
        clone_state_dict(tail.state_dict()),
        total_loss / max(1, steps),
    )


@torch.no_grad()
def evaluate_clean(
    head: nn.Module,
    backbone: nn.Module,
    tail: nn.Module,
    loader: DataLoader,
    device: str,
) -> float:
    head.eval()
    backbone.eval()
    tail.eval()

    correct = 0
    total = 0
    for x, y in loader:
        x = x.to(device)
        y = y.to(device)
        logits = tail(backbone(head(x)))
        pred = logits.argmax(dim=1)
        correct += int((pred == y).sum().item())
        total += int(y.numel())
    return 100.0 * correct / max(1, total)


@torch.no_grad()
def evaluate_ba(
    head: nn.Module,
    backbone: nn.Module,
    tail: nn.Module,
    loader: DataLoader,
    target_label: int,
    trigger_size: int,
    trigger_value: float,
    device: str,
) -> float:
    head.eval()
    backbone.eval()
    tail.eval()

    success = 0
    total = 0
    for x, y in loader:
        x = add_pixel_trigger(x, trigger_size, trigger_value).to(device)
        pred = tail(backbone(head(x))).argmax(dim=1).cpu()
        success += int((pred == target_label).sum().item())
        total += int(pred.numel())

    return 100.0 * success / max(1, total)


def evaluate_asr(
    head: nn.Module,
    backbone: nn.Module,
    tail: nn.Module,
    loader: DataLoader,
    target_label: int,
    trigger_size: int,
    trigger_value: float,
    device: str,
) -> float:
    head.eval()
    backbone.eval()
    tail.eval()

    success = 0
    total = 0
    for x, y in loader:
        mask = y != target_label
        x = x[mask]
        if x.numel() == 0:
            continue

        x = add_pixel_trigger(x, trigger_size, trigger_value).to(device)
        pred = tail(backbone(head(x))).argmax(dim=1).cpu()

        success += int((pred == target_label).sum().item())
        total += int(pred.numel())

    return 100.0 * success / max(1, total)


# ============================================================
# SafeSplit
# ============================================================

def majority_size(n: int) -> int:
    return n // 2 + 1


def sum_smallest(values: List[float], k: int) -> float:
    if not values:
        return 0.0
    values = sorted(values)
    return float(sum(values[:k]))


def select_latest_benign_checkpoint(
    history: List[Dict[str, Dict[str, torch.Tensor]]],
    cfg: Config,
) -> int:
    t = len(history) - 1
    if t <= 0:
        return 0

    start = max(1, t - cfg.clients + 1)
    idxs = list(range(start, t + 1))
    m = len(idxs)
    maj = majority_size(m)

    delta_vecs: Dict[int, torch.Tensor] = {}
    freq_sigs: Dict[int, torch.Tensor] = {}

    for i in idxs:
        delta = subtract_state_dict(history[i]["backbone"], history[i - 1]["backbone"])
        delta_vecs[i] = flatten_state_dict(delta)
        freq_sigs[i] = low_frequency_signature(
            delta,
            keep_ratio=cfg.low_freq_keep_ratio,
            min_keep_per_tensor=cfg.min_keep_per_tensor,
            max_keep_per_tensor=cfg.max_keep_per_tensor,
        )

    E = {}
    for i in idxs:
        distances = []
        for j in idxs:
            if i == j:
                continue
            a = freq_sigs[i]
            b = freq_sigs[j]
            if a.numel() != b.numel():
                min_len = min(a.numel(), b.numel())
                a = a[:min_len]
                b = b[:min_len]
            d = torch.norm(a - b, p=2).item()
            distances.append(float(d))
        E[i] = sum_smallest(distances, min(maj, len(distances)))

    freq_majority = set(sorted(idxs, key=lambda i: E[i])[:maj])

    if cfg.use_rotation:
        ordered_delta_vecs = [delta_vecs[i] for i in idxs]
        rot_scores_list = build_rotation_scores(ordered_delta_vecs)
        R = {idxs[pos]: rot_scores_list[pos] for pos in range(len(idxs))}
        rot_majority = set(sorted(idxs, key=lambda i: R[i])[:maj])
        benign_majority = freq_majority.intersection(rot_majority)
    else:
        benign_majority = freq_majority

    if not benign_majority:
        return max(0, t - 1)

    for i in reversed(idxs):
        if i in benign_majority:
            return i

    return max(0, t - 1)


# ============================================================
# Main
# ============================================================

def run_experiment(cfg: Config) -> None:
    set_seed(cfg.seed)
    os.makedirs(cfg.save_dir, exist_ok=True)

    metrics_path = os.path.join(cfg.save_dir, "metrics.csv")
    with open(metrics_path, "w", newline="") as f:
        writer = csv.writer(f)
        writer.writerow(["round", "ma", "asr", "ba"])

    client_loaders, clean_test_loader = build_dataloaders(cfg)

    head, backbone, tail = build_split_resnet18(num_classes=10)
    head = head.to(cfg.device)
    backbone = backbone.to(cfg.device)
    tail = tail.to(cfg.device)

    print("\n========== Split Parameter Check ==========")
    print(f"head params     : {count_trainable_params(head):,}")
    print(f"backbone params : {count_trainable_params(backbone):,}")
    print(f"tail params     : {count_trainable_params(tail):,}")
    print(f"total params    : {count_trainable_params(head) + count_trainable_params(backbone) + count_trainable_params(tail):,}")

    current_head_state = clone_state_dict(head.state_dict())
    current_backbone_state = clone_state_dict(backbone.state_dict())
    current_tail_state = clone_state_dict(tail.state_dict())

    history: List[Dict[str, Dict[str, torch.Tensor]]] = [{
        "head": clone_state_dict(current_head_state),
        "backbone": clone_state_dict(current_backbone_state),
        "tail": clone_state_dict(current_tail_state),
    }]

    total_steps = cfg.rounds * cfg.clients
    malicious_ids = set(parse_malicious_client_ids(cfg))

    for step in range(1, total_steps + 1):
        cid = (step - 1) % cfg.clients
        is_malicious = cid in malicious_ids

        load_state_dict_to_module(head, current_head_state, cfg.device)
        load_state_dict_to_module(backbone, current_backbone_state, cfg.device)
        load_state_dict_to_module(tail, current_tail_state, cfg.device)

        local_epochs = cfg.malicious_local_epochs if is_malicious else cfg.local_epochs

        h_state, b_state, t_state, avg_loss = train_one_client(
            head, backbone, tail, client_loaders[cid], cfg, local_epochs=local_epochs
        )

        history.append({
            "head": h_state,
            "backbone": b_state,
            "tail": t_state,
        })

        if cfg.defense:
            chosen_idx = select_latest_benign_checkpoint(history, cfg)
        else:
            chosen_idx = len(history) - 1

        flagged = chosen_idx != len(history) - 1

        current_head_state = clone_state_dict(history[chosen_idx]["head"])
        current_backbone_state = clone_state_dict(history[chosen_idx]["backbone"])
        current_tail_state = clone_state_dict(history[chosen_idx]["tail"])

        print(
            f"Step {step:03d}/{total_steps} | client={cid:02d} | "
            f"{'malicious' if is_malicious else 'benign':9s} | "
            f"epochs={local_epochs} | loss={avg_loss:.4f} | "
            f"chosen_checkpoint={chosen_idx} | "
            f"{'ROLLBACK' if flagged else 'accepted'}"
        )

        if step % cfg.clients == 0:
            round_id = step // cfg.clients

            load_state_dict_to_module(head, current_head_state, cfg.device)
            load_state_dict_to_module(backbone, current_backbone_state, cfg.device)
            load_state_dict_to_module(tail, current_tail_state, cfg.device)

            ma = evaluate_clean(head, backbone, tail, clean_test_loader, cfg.device)
            asr = evaluate_asr(
                head,
                backbone,
                tail,
                clean_test_loader,
                cfg.target_label,
                cfg.trigger_size,
                cfg.trigger_value,
                cfg.device,
            )
            ba = evaluate_ba(
                head,
                backbone,
                tail,
                clean_test_loader,
                cfg.target_label,
                cfg.trigger_size,
                cfg.trigger_value,
                cfg.device,
            )

            print(f"[Round {round_id:02d}] MA={ma:.2f}% | ASR={asr:.2f}% | BA={ba:.2f}%")

            with open(metrics_path, "a", newline="") as f:
                writer = csv.writer(f)
                writer.writerow([round_id, f"{ma:.4f}", f"{asr:.4f}", f"{ba:.4f}"])

            torch.save(
                {
                    "round": round_id,
                    "head": current_head_state,
                    "backbone": current_backbone_state,
                    "tail": current_tail_state,
                    "ma": ma,
                    "asr": asr,
                    "ba": ba,
                    "config": vars(cfg),
                },
                os.path.join(cfg.save_dir, f"checkpoint_round_{round_id:02d}.pt"),
            )

    load_state_dict_to_module(head, current_head_state, cfg.device)
    load_state_dict_to_module(backbone, current_backbone_state, cfg.device)
    load_state_dict_to_module(tail, current_tail_state, cfg.device)

    final_ma = evaluate_clean(head, backbone, tail, clean_test_loader, cfg.device)
    final_asr = evaluate_asr(
        head,
        backbone,
        tail,
        clean_test_loader,
        cfg.target_label,
        cfg.trigger_size,
        cfg.trigger_value,
        cfg.device,
    )
    final_ba = evaluate_ba(
        head,
        backbone,
        tail,
        clean_test_loader,
        cfg.target_label,
        cfg.trigger_size,
        cfg.trigger_value,
        cfg.device,
    )

    final_path = os.path.join(cfg.save_dir, "final_results.txt")
    with open(final_path, "w") as f:
        f.write(f"Main Task Accuracy (MA): {final_ma:.4f}\n")
        f.write(f"Attack Success Rate (ASR): {final_asr:.4f}\n")
        f.write(f"Backdoor Accuracy (BA): {final_ba:.4f}\n")
        f.write(f"Config: {vars(cfg)}\n")
        f.write("Notes: ASR excludes original target-label samples; BA evaluates the entire triggered test set. target-label training samples are excluded from poisoning.\n")
        f.write("Paper target used here: CIFAR-10 Pixel Trigger, Table II direction: No Defense BA≈100.0/MA≈66.6; SafeSplit BA≈0.3/MA≈66.4.\n")
        f.write("Paper-aligned knobs: clients=10, malicious_clients=2, iid_rate=0.8, PDR/poison_fraction=0.75, red 4x4 bottom-right pixel trigger, target_label=2, augmentation ON.\n")

    print("\n========== Final ==========")
    print(f"Main Task Accuracy (MA): {final_ma:.2f}%")
    print(f"Attack Success Rate (ASR): {final_asr:.2f}%")
    print(f"Backdoor Accuracy (BA): {final_ba:.2f}%")
    print(f"Saved metrics to: {metrics_path}")
    print(f"Saved final results to: {final_path}")


def parse_args() -> Config:
    parser = argparse.ArgumentParser(description="SafeSplit-style split learning backdoor experiment on CIFAR-10")

    parser.add_argument("--data_root", type=str, default="./data")
    parser.add_argument("--batch_size", type=int, default=128)
    parser.add_argument("--num_workers", type=int, default=2)
    parser.add_argument("--rounds", type=int, default=50)
    parser.add_argument("--clients", type=int, default=10)
    parser.add_argument("--malicious_clients", type=int, default=2)
    parser.add_argument("--malicious_client_ids", type=str, default="8,9")
    parser.add_argument("--local_epochs", type=int, default=1)
    parser.add_argument("--malicious_local_epochs", type=int, default=2)
    parser.add_argument("--lr", type=float, default=0.001)
    parser.add_argument("--momentum", type=float, default=0.9)
    parser.add_argument("--weight_decay", type=float, default=1e-4)
    parser.add_argument("--seed", type=int, default=0)
    parser.add_argument("--iid_rate", type=float, default=0.8)
    parser.add_argument("--poison_fraction", type=float, default=0.75)

    parser.add_argument(
        "--no_train_augmentation",
        action="store_true",
        help="Disable CIFAR-10 RandomCrop/RandomHorizontalFlip. Default: augmentation ON for higher MA.",
    )

    parser.add_argument("--target_label", type=int, default=2)
    parser.add_argument("--trigger_size", type=int, default=4)
    parser.add_argument("--trigger_value", type=float, default=1.0)

    # Default is now No Defense.
    # Use --defense only when you want to enable SafeSplit.
    parser.add_argument(
        "--defense",
        action="store_true",
        help="Enable SafeSplit defense. Default: defense OFF for no-defense baseline.",
    )

    parser.add_argument(
        "--no_rotation",
        action="store_true",
        help="Disable rotational analysis. Default: rotation ON if defense is enabled.",
    )

    parser.add_argument("--low_freq_keep_ratio", type=float, default=0.02)
    parser.add_argument("--min_keep_per_tensor", type=int, default=2)
    parser.add_argument("--max_keep_per_tensor", type=int, default=8)
    parser.add_argument("--device", type=str, default="cuda" if torch.cuda.is_available() else "cpu")
    parser.add_argument("--save_dir", type=str, default="./outputs_no_defense")

    # Colab/Jupyter compatible
    args, unknown = parser.parse_known_args()

    cfg = Config(
        data_root=args.data_root,
        batch_size=args.batch_size,
        num_workers=args.num_workers,
        rounds=args.rounds,
        clients=args.clients,
        malicious_clients=args.malicious_clients,
        malicious_client_ids=args.malicious_client_ids,
        local_epochs=args.local_epochs,
        malicious_local_epochs=args.malicious_local_epochs,
        lr=args.lr,
        momentum=args.momentum,
        weight_decay=args.weight_decay,
        seed=args.seed,
        iid_rate=args.iid_rate,
        poison_fraction=args.poison_fraction,
        train_augmentation=not args.no_train_augmentation,
        target_label=args.target_label,
        trigger_size=args.trigger_size,
        trigger_value=args.trigger_value,
        defense=args.defense,
        use_rotation=not args.no_rotation,
        low_freq_keep_ratio=args.low_freq_keep_ratio,
        min_keep_per_tensor=args.min_keep_per_tensor,
        max_keep_per_tensor=args.max_keep_per_tensor,
        device=args.device,
        save_dir=args.save_dir,
    )
    return cfg


if __name__ == "__main__":
    cfg = parse_args()
    print(f"Initial Config: {cfg}")
    print("Hint: paper-aligned default = CIFAR-10 pixel trigger, PDR=0.75, iid_rate=0.8, clients=10, malicious=2, SafeSplit defense ON, rotation ON, augmentation ON. Use --no_defense for baseline.")
    run_experiment(cfg)


Initial Config: Config(data_root='./data', batch_size=128, num_workers=2, rounds=50, clients=10, malicious_clients=2, malicious_client_ids='8,9', local_epochs=1, malicious_local_epochs=2, lr=0.001, momentum=0.9, weight_decay=0.0001, seed=0, iid_rate=0.8, poison_fraction=0.75, train_augmentation=True, target_label=2, trigger_size=4, trigger_value=1.0, defense=False, use_rotation=True, low_freq_keep_ratio=0.02, min_keep_per_tensor=2, max_keep_per_tensor=8, device='cuda', save_dir='./outputs_no_defense')
Hint: paper-aligned default = CIFAR-10 pixel trigger, PDR=0.75, iid_rate=0.8, clients=10, malicious=2, SafeSplit defense ON, rotation ON, augmentation ON. Use --no_defense for baseline.

========== Split Parameter Check ==========
head params     : 9,536
backbone params : 11,166,976
tail params     : 5,130
total params    : 11,181,642
Step 001/500 | client=00 | benign    | epochs=1 | loss=2.1288 | chosen_checkpoint=1 | accepted
Step 002/500 | client=01 | benign    | epochs=1 | loss=1.9975

In [ ]:
import argparse
import csv
import math
import os
import random
import sys
from dataclasses import dataclass
from typing import Dict, List, Tuple

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset, Subset


# ============================================================
# Safe torchvision import
# ============================================================

def safe_import_torchvision():
    try:
        from torchvision import datasets, transforms, models
        return datasets, transforms, models
    except Exception:
        for name in list(sys.modules.keys()):
            if name.startswith("torchvision"):
                del sys.modules[name]

        from torch.library import Library
        lib = Library("torchvision", "DEF")
        try:
            lib.define("nms(Tensor dets, Tensor scores, float iou_threshold) -> Tensor")
        except Exception:
            pass

        from torchvision import datasets, transforms, models
        return datasets, transforms, models


datasets, transforms, models = safe_import_torchvision()


# ============================================================
# Utils
# ============================================================

def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    if hasattr(torch.backends, 'cudnn'):
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False


def clone_state_dict(state_dict: Dict[str, torch.Tensor]) -> Dict[str, torch.Tensor]:
    return {k: v.detach().cpu().clone() for k, v in state_dict.items()}


def load_state_dict_to_module(module: nn.Module, state_dict: Dict[str, torch.Tensor], device: str) -> None:
    module.load_state_dict({k: v.to(device) for k, v in state_dict.items()})


def subtract_state_dict(
    a: Dict[str, torch.Tensor],
    b: Dict[str, torch.Tensor],
) -> Dict[str, torch.Tensor]:
    return {k: a[k].float() - b[k].float() for k in a.keys()}


def flatten_state_dict(state_dict: Dict[str, torch.Tensor]) -> torch.Tensor:
    parts = []
    for v in state_dict.values():
        parts.append(v.reshape(-1).float().cpu())
    if not parts:
        return torch.zeros(1)
    return torch.cat(parts)


def count_trainable_params(module: nn.Module) -> int:
    return sum(p.numel() for p in module.parameters() if p.requires_grad)


# ============================================================
# DCT utilities
# ============================================================

def dct_1d_torch(x: torch.Tensor) -> torch.Tensor:
    x = x.flatten().float().cpu()
    n = x.shape[0]
    if n == 0:
        return x
    if n == 1:
        return x.clone()

    v = torch.cat([x[::2], x[1::2].flip(0)])
    Vc = torch.fft.fft(v)

    k = torch.arange(n, dtype=torch.float32)
    factor = -math.pi * k / (2.0 * n)
    W_r = torch.cos(factor)
    W_i = torch.sin(factor)

    out = 2.0 * (Vc[:n].real * W_r - Vc[:n].imag * W_i)
    return out


def dct_2d_torch(x: torch.Tensor) -> torch.Tensor:
    x = x.float().cpu()
    if x.ndim != 2:
        raise ValueError("dct_2d_torch expects a 2D tensor.")

    rows = torch.stack([dct_1d_torch(row) for row in x], dim=0)
    cols = torch.stack([dct_1d_torch(col) for col in rows.transpose(0, 1)], dim=0)
    return cols.transpose(0, 1).contiguous()


def lowfreq_block_2d(
    coeff: torch.Tensor,
    keep_ratio: float,
    min_keep: int,
    max_keep: int,
) -> torch.Tensor:
    h, w = coeff.shape
    kh = max(min_keep, int(h * keep_ratio))
    kw = max(min_keep, int(w * keep_ratio))
    kh = min(kh, max_keep, h)
    kw = min(kw, max_keep, w)
    return coeff[:kh, :kw].reshape(-1)


def low_frequency_signature(
    delta_state: Dict[str, torch.Tensor],
    keep_ratio: float = 0.02,
    min_keep_per_tensor: int = 2,
    max_keep_per_tensor: int = 8,
) -> torch.Tensor:
    parts = []

    for _, v in delta_state.items():
        x = v.detach().float().cpu()

        if x.ndim == 4:
            per_kernel_parts = []
            for o in range(x.shape[0]):
                for i in range(x.shape[1]):
                    coeff = dct_2d_torch(x[o, i])
                    per_kernel_parts.append(
                        lowfreq_block_2d(
                            coeff,
                            keep_ratio=keep_ratio,
                            min_keep=min_keep_per_tensor,
                            max_keep=max_keep_per_tensor,
                        )
                    )
            if per_kernel_parts:
                parts.append(torch.cat(per_kernel_parts))

        elif x.ndim == 2:
            coeff = dct_2d_torch(x)
            parts.append(
                lowfreq_block_2d(
                    coeff,
                    keep_ratio=keep_ratio,
                    min_keep=min_keep_per_tensor,
                    max_keep=max_keep_per_tensor,
                )
            )

        elif x.ndim == 1:
            coeff = dct_1d_torch(x)
            keep = max(min_keep_per_tensor, int(coeff.numel() * keep_ratio))
            keep = min(keep, max_keep_per_tensor, coeff.numel())
            if keep > 0:
                parts.append(coeff[:keep])

        else:
            flat = x.reshape(-1)
            coeff = dct_1d_torch(flat)
            keep = max(min_keep_per_tensor, int(coeff.numel() * keep_ratio))
            keep = min(keep, max_keep_per_tensor, coeff.numel())
            if keep > 0:
                parts.append(coeff[:keep])

    if not parts:
        return torch.zeros(1)
    return torch.cat(parts)


# ============================================================
# Dynamic / rotation utilities
# ============================================================

def rotational_feature(delta_vec: torch.Tensor) -> torch.Tensor:
    return torch.atan(delta_vec.float().cpu())


def build_rotation_scores(delta_vecs_in_order: List[torch.Tensor]) -> List[float]:
    if len(delta_vecs_in_order) == 0:
        return []

    theta = [rotational_feature(d) for d in delta_vecs_in_order]

    omega = []
    for i in range(len(theta)):
        if i == 0:
            omega.append(theta[i] / (2.0 * math.pi))
        else:
            omega.append((theta[i] - theta[i - 1]) / (2.0 * math.pi))

    n = len(omega)
    maj = n // 2 + 1
    scores = []

    for i in range(n):
        distances = []
        for j in range(n):
            if i == j:
                continue
            d = torch.norm(omega[i] - omega[j], p=2).item()
            distances.append(float(d))
        distances = sorted(distances)
        scores.append(float(sum(distances[:min(maj, len(distances))])))

    return scores


# ============================================================
# Config
# ============================================================

@dataclass
class Config:
    data_root: str = "./data"
    batch_size: int = 128
    num_workers: int = 2
    rounds: int = 50
    clients: int = 10
    malicious_clients: int = 2
    malicious_client_ids: str = "8,9"
    local_epochs: int = 1
    malicious_local_epochs: int = 2
    lr: float = 0.001
    momentum: float = 0.9
    weight_decay: float = 1e-4
    seed: int = 0
    iid_rate: float = 0.8
    poison_fraction: float = 0.75
    train_augmentation: bool = True
    target_label: int = 2
    trigger_size: int = 4
    trigger_value: float = 1.0
    defense: bool = True
    use_rotation: bool = True
    low_freq_keep_ratio: float = 0.02
    min_keep_per_tensor: int = 2
    max_keep_per_tensor: int = 8
    device: str = "cuda" if torch.cuda.is_available() else "cpu"
    save_dir: str = "./outputs_paper_aligned"


# ============================================================
# Trigger / poisoned dataset
# ============================================================

def add_pixel_trigger(images: torch.Tensor, trigger_size: int, value: float) -> torch.Tensor:
    x = images.clone()
    # CIFAR-10 tensors are normalized before the trigger is applied.
    # So we write the *normalized* equivalent of a bright red patch:
    # raw RGB = (value, 0, 0) -> normalized channel values.
    cifar10_mean = (0.4914, 0.4822, 0.4465)
    cifar10_std = (0.2470, 0.2435, 0.2616)

    r = (value - cifar10_mean[0]) / cifar10_std[0]
    g = (0.0 - cifar10_mean[1]) / cifar10_std[1]
    b = (0.0 - cifar10_mean[2]) / cifar10_std[2]

    x[:, 0, -trigger_size:, -trigger_size:] = r
    if x.shape[1] > 1:
        x[:, 1, -trigger_size:, -trigger_size:] = g
    if x.shape[1] > 2:
        x[:, 2, -trigger_size:, -trigger_size:] = b
    return x


class PoisonedSubset(Dataset):
    def __init__(
        self,
        base_dataset: Dataset,
        indices: List[int],
        poison_fraction: float,
        target_label: int,
        trigger_size: int,
        trigger_value: float,
        train: bool = True,
        seed: int = 0,
    ) -> None:
        self.base_dataset = base_dataset
        self.indices = list(indices)
        self.poison_fraction = poison_fraction
        self.target_label = target_label
        self.trigger_size = trigger_size
        self.trigger_value = trigger_value
        self.train = train

        rng = random.Random(seed)
        eligible_positions = []
        for pos, ds_idx in enumerate(self.indices):
            y = int(getattr(self.base_dataset, 'targets')[ds_idx])
            if y != self.target_label:
                eligible_positions.append(pos)

        if train:
            n_poison = int(len(eligible_positions) * poison_fraction)
            rng.shuffle(eligible_positions)
            self.poison_positions = set(eligible_positions[:n_poison])
        else:
            self.poison_positions = set(eligible_positions)

    def __len__(self) -> int:
        return len(self.indices)

    def __getitem__(self, idx: int):
        x, y = self.base_dataset[self.indices[idx]]
        if idx in self.poison_positions:
            x = add_pixel_trigger(x.unsqueeze(0), self.trigger_size, self.trigger_value).squeeze(0)
            y = self.target_label
        return x, y


# ============================================================
# Partition
# ============================================================

def partition_main_label_non_iid(
    targets: List[int],
    num_clients: int,
    iid_rate: float,
    seed: int,
) -> Dict[int, List[int]]:
    rng = random.Random(seed)
    num_classes = len(set(int(y) for y in targets))
    all_indices = list(range(len(targets)))
    rng.shuffle(all_indices)

    total_per_client = len(targets) // num_clients
    iid_per_client = int(total_per_client * iid_rate)
    non_iid_per_client = total_per_client - iid_per_client

    client_to_indices: Dict[int, List[int]] = {i: [] for i in range(num_clients)}

    # 1) IID part without replacement
    ptr = 0
    used = set()
    for cid in range(num_clients):
        chosen = all_indices[ptr:ptr + iid_per_client]
        client_to_indices[cid].extend(chosen)
        used.update(chosen)
        ptr += iid_per_client

    # 2) Remaining samples are grouped by label
    label_to_indices: Dict[int, List[int]] = {c: [] for c in range(num_classes)}
    for idx, y in enumerate(targets):
        if idx not in used:
            label_to_indices[int(y)].append(idx)

    for lst in label_to_indices.values():
        rng.shuffle(lst)

    # 3) Assign one main label per client
    main_labels = [i % num_classes for i in range(num_clients)]
    rng.shuffle(main_labels)

    for cid in range(num_clients):
        label = main_labels[cid]
        take = min(non_iid_per_client, len(label_to_indices[label]))
        chosen = label_to_indices[label][:take]
        label_to_indices[label] = label_to_indices[label][take:]
        client_to_indices[cid].extend(chosen)

    # 4) Fill any shortages from leftovers without replacement
    leftovers = []
    for lst in label_to_indices.values():
        leftovers.extend(lst)
    rng.shuffle(leftovers)

    p = 0
    for cid in range(num_clients):
        while len(client_to_indices[cid]) < total_per_client and p < len(leftovers):
            client_to_indices[cid].append(leftovers[p])
            p += 1

    return client_to_indices




def parse_malicious_client_ids(cfg: Config) -> List[int]:
    if cfg.malicious_client_ids.strip():
        ids = [int(x.strip()) for x in cfg.malicious_client_ids.split(',') if x.strip()]
        ids = sorted(set(i for i in ids if 0 <= i < cfg.clients))
        if ids:
            return ids
    # Fallback: use the last K clients so the backdoor is not immediately washed out by many benign clients
    start = max(0, cfg.clients - cfg.malicious_clients)
    return list(range(start, cfg.clients))

# ============================================================
# ResNet18 split aligned closer to paper Table I
# head params ~= 9536, tail params ~= 5130 for CIFAR-10
# ============================================================

class FlattenTail(nn.Module):
    def __init__(self, fc: nn.Module):
        super().__init__()
        self.fc = fc

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = torch.flatten(x, 1)
        x = self.fc(x)
        return x


def build_split_resnet18(num_classes: int = 10) -> Tuple[nn.Module, nn.Module, nn.Module]:
    model = models.resnet18(weights=None, num_classes=num_classes)

    head = nn.Sequential(
        model.conv1,
        model.bn1,
        model.relu,
        model.maxpool,
    )

    backbone = nn.Sequential(
        model.layer1,
        model.layer2,
        model.layer3,
        model.layer4,
        model.avgpool,
    )

    tail = FlattenTail(model.fc)
    return head, backbone, tail


# ============================================================
# Data
# ============================================================

def build_dataloaders(cfg: Config):
    cifar10_mean = (0.4914, 0.4822, 0.4465)
    cifar10_std = (0.2470, 0.2435, 0.2616)

    if cfg.train_augmentation:
        transform_train = transforms.Compose([
            transforms.RandomCrop(32, padding=4),
            transforms.RandomHorizontalFlip(),
            transforms.ToTensor(),
            transforms.Normalize(cifar10_mean, cifar10_std),
        ])
    else:
        transform_train = transforms.Compose([
            transforms.ToTensor(),
            transforms.Normalize(cifar10_mean, cifar10_std),
        ])
    transform_test = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize(cifar10_mean, cifar10_std),
    ])

    train_set = datasets.CIFAR10(
        root=cfg.data_root,
        train=True,
        download=True,
        transform=transform_train,
    )
    test_set = datasets.CIFAR10(
        root=cfg.data_root,
        train=False,
        download=True,
        transform=transform_test,
    )

    partitions = partition_main_label_non_iid(train_set.targets, cfg.clients, cfg.iid_rate, cfg.seed)
    malicious_ids = set(parse_malicious_client_ids(cfg))

    client_loaders = {}
    for cid, idxs in partitions.items():
        if cid in malicious_ids:
            ds = PoisonedSubset(
                train_set,
                idxs,
                poison_fraction=cfg.poison_fraction,
                target_label=cfg.target_label,
                trigger_size=cfg.trigger_size,
                trigger_value=cfg.trigger_value,
                train=True,
                seed=cfg.seed + cid,
            )
        else:
            ds = Subset(train_set, idxs)

        client_loaders[cid] = DataLoader(
            ds,
            batch_size=cfg.batch_size,
            shuffle=True,
            num_workers=cfg.num_workers,
            pin_memory=torch.cuda.is_available(),
        )

    clean_test_loader = DataLoader(
        test_set,
        batch_size=cfg.batch_size,
        shuffle=False,
        num_workers=cfg.num_workers,
        pin_memory=torch.cuda.is_available(),
    )

    return client_loaders, clean_test_loader


# ============================================================
# Train / eval
# ============================================================

def make_optimizer(module: nn.Module, cfg: Config) -> torch.optim.Optimizer:
    return torch.optim.SGD(
        module.parameters(),
        lr=cfg.lr,
        momentum=cfg.momentum,
        weight_decay=cfg.weight_decay,
    )


def train_one_client(
    head: nn.Module,
    backbone: nn.Module,
    tail: nn.Module,
    loader: DataLoader,
    cfg: Config,
    local_epochs: int,
) -> Tuple[Dict[str, torch.Tensor], Dict[str, torch.Tensor], Dict[str, torch.Tensor], float]:
    head.train()
    backbone.train()
    tail.train()

    head_opt = make_optimizer(head, cfg)
    back_opt = make_optimizer(backbone, cfg)
    tail_opt = make_optimizer(tail, cfg)

    total_loss = 0.0
    steps = 0

    for _ in range(local_epochs):
        for x, y in loader:
            x = x.to(cfg.device)
            y = y.to(cfg.device)

            head_opt.zero_grad()
            back_opt.zero_grad()
            tail_opt.zero_grad()

            smashed = head(x)
            features = backbone(smashed)
            logits = tail(features)

            loss = F.cross_entropy(logits, y)
            loss.backward()

            head_opt.step()
            back_opt.step()
            tail_opt.step()

            total_loss += float(loss.item())
            steps += 1

    return (
        clone_state_dict(head.state_dict()),
        clone_state_dict(backbone.state_dict()),
        clone_state_dict(tail.state_dict()),
        total_loss / max(1, steps),
    )


@torch.no_grad()
def evaluate_clean(
    head: nn.Module,
    backbone: nn.Module,
    tail: nn.Module,
    loader: DataLoader,
    device: str,
) -> float:
    head.eval()
    backbone.eval()
    tail.eval()

    correct = 0
    total = 0
    for x, y in loader:
        x = x.to(device)
        y = y.to(device)
        logits = tail(backbone(head(x)))
        pred = logits.argmax(dim=1)
        correct += int((pred == y).sum().item())
        total += int(y.numel())
    return 100.0 * correct / max(1, total)


@torch.no_grad()
def evaluate_ba(
    head: nn.Module,
    backbone: nn.Module,
    tail: nn.Module,
    loader: DataLoader,
    target_label: int,
    trigger_size: int,
    trigger_value: float,
    device: str,
) -> float:
    head.eval()
    backbone.eval()
    tail.eval()

    success = 0
    total = 0
    for x, y in loader:
        x = add_pixel_trigger(x, trigger_size, trigger_value).to(device)
        pred = tail(backbone(head(x))).argmax(dim=1).cpu()
        success += int((pred == target_label).sum().item())
        total += int(pred.numel())

    return 100.0 * success / max(1, total)


def evaluate_asr(
    head: nn.Module,
    backbone: nn.Module,
    tail: nn.Module,
    loader: DataLoader,
    target_label: int,
    trigger_size: int,
    trigger_value: float,
    device: str,
) -> float:
    head.eval()
    backbone.eval()
    tail.eval()

    success = 0
    total = 0
    for x, y in loader:
        mask = y != target_label
        x = x[mask]
        if x.numel() == 0:
            continue

        x = add_pixel_trigger(x, trigger_size, trigger_value).to(device)
        pred = tail(backbone(head(x))).argmax(dim=1).cpu()

        success += int((pred == target_label).sum().item())
        total += int(pred.numel())

    return 100.0 * success / max(1, total)


# ============================================================
# SafeSplit
# ============================================================

def majority_size(n: int) -> int:
    return n // 2 + 1


def sum_smallest(values: List[float], k: int) -> float:
    if not values:
        return 0.0
    values = sorted(values)
    return float(sum(values[:k]))


def select_latest_benign_checkpoint(
    history: List[Dict[str, Dict[str, torch.Tensor]]],
    cfg: Config,
) -> int:
    t = len(history) - 1
    if t <= 0:
        return 0

    start = max(1, t - cfg.clients + 1)
    idxs = list(range(start, t + 1))
    m = len(idxs)
    maj = majority_size(m)

    delta_vecs: Dict[int, torch.Tensor] = {}
    freq_sigs: Dict[int, torch.Tensor] = {}

    for i in idxs:
        delta = subtract_state_dict(history[i]["backbone"], history[i - 1]["backbone"])
        delta_vecs[i] = flatten_state_dict(delta)
        freq_sigs[i] = low_frequency_signature(
            delta,
            keep_ratio=cfg.low_freq_keep_ratio,
            min_keep_per_tensor=cfg.min_keep_per_tensor,
            max_keep_per_tensor=cfg.max_keep_per_tensor,
        )

    E = {}
    for i in idxs:
        distances = []
        for j in idxs:
            if i == j:
                continue
            a = freq_sigs[i]
            b = freq_sigs[j]
            if a.numel() != b.numel():
                min_len = min(a.numel(), b.numel())
                a = a[:min_len]
                b = b[:min_len]
            d = torch.norm(a - b, p=2).item()
            distances.append(float(d))
        E[i] = sum_smallest(distances, min(maj, len(distances)))

    freq_majority = set(sorted(idxs, key=lambda i: E[i])[:maj])

    if cfg.use_rotation:
        ordered_delta_vecs = [delta_vecs[i] for i in idxs]
        rot_scores_list = build_rotation_scores(ordered_delta_vecs)
        R = {idxs[pos]: rot_scores_list[pos] for pos in range(len(idxs))}
        rot_majority = set(sorted(idxs, key=lambda i: R[i])[:maj])
        benign_majority = freq_majority.intersection(rot_majority)
    else:
        benign_majority = freq_majority

    if not benign_majority:
        return max(0, t - 1)

    for i in reversed(idxs):
        if i in benign_majority:
            return i

    return max(0, t - 1)


# ============================================================
# Main
# ============================================================

def run_experiment(cfg: Config) -> None:
    set_seed(cfg.seed)
    os.makedirs(cfg.save_dir, exist_ok=True)

    metrics_path = os.path.join(cfg.save_dir, "metrics.csv")
    with open(metrics_path, "w", newline="") as f:
        writer = csv.writer(f)
        writer.writerow(["round", "ma", "asr", "ba"])

    client_loaders, clean_test_loader = build_dataloaders(cfg)

    head, backbone, tail = build_split_resnet18(num_classes=10)
    head = head.to(cfg.device)
    backbone = backbone.to(cfg.device)
    tail = tail.to(cfg.device)

    print("\n========== Split Parameter Check ==========")
    print(f"head params     : {count_trainable_params(head):,}")
    print(f"backbone params : {count_trainable_params(backbone):,}")
    print(f"tail params     : {count_trainable_params(tail):,}")
    print(f"total params    : {count_trainable_params(head) + count_trainable_params(backbone) + count_trainable_params(tail):,}")

    current_head_state = clone_state_dict(head.state_dict())
    current_backbone_state = clone_state_dict(backbone.state_dict())
    current_tail_state = clone_state_dict(tail.state_dict())

    history: List[Dict[str, Dict[str, torch.Tensor]]] = [{
        "head": clone_state_dict(current_head_state),
        "backbone": clone_state_dict(current_backbone_state),
        "tail": clone_state_dict(current_tail_state),
    }]

    total_steps = cfg.rounds * cfg.clients
    malicious_ids = set(parse_malicious_client_ids(cfg))

    for step in range(1, total_steps + 1):
        cid = (step - 1) % cfg.clients
        is_malicious = cid in malicious_ids

        load_state_dict_to_module(head, current_head_state, cfg.device)
        load_state_dict_to_module(backbone, current_backbone_state, cfg.device)
        load_state_dict_to_module(tail, current_tail_state, cfg.device)

        local_epochs = cfg.malicious_local_epochs if is_malicious else cfg.local_epochs

        h_state, b_state, t_state, avg_loss = train_one_client(
            head, backbone, tail, client_loaders[cid], cfg, local_epochs=local_epochs
        )

        history.append({
            "head": h_state,
            "backbone": b_state,
            "tail": t_state,
        })

        if cfg.defense:
            chosen_idx = select_latest_benign_checkpoint(history, cfg)
        else:
            chosen_idx = len(history) - 1

        flagged = chosen_idx != len(history) - 1

        current_head_state = clone_state_dict(history[chosen_idx]["head"])
        current_backbone_state = clone_state_dict(history[chosen_idx]["backbone"])
        current_tail_state = clone_state_dict(history[chosen_idx]["tail"])

        print(
            f"Step {step:03d}/{total_steps} | client={cid:02d} | "
            f"{'malicious' if is_malicious else 'benign':9s} | "
            f"epochs={local_epochs} | loss={avg_loss:.4f} | "
            f"chosen_checkpoint={chosen_idx} | "
            f"{'ROLLBACK' if flagged else 'accepted'}"
        )

        if step % cfg.clients == 0:
            round_id = step // cfg.clients

            load_state_dict_to_module(head, current_head_state, cfg.device)
            load_state_dict_to_module(backbone, current_backbone_state, cfg.device)
            load_state_dict_to_module(tail, current_tail_state, cfg.device)

            ma = evaluate_clean(head, backbone, tail, clean_test_loader, cfg.device)
            asr = evaluate_asr(
                head,
                backbone,
                tail,
                clean_test_loader,
                cfg.target_label,
                cfg.trigger_size,
                cfg.trigger_value,
                cfg.device,
            )
            ba = evaluate_ba(
                head,
                backbone,
                tail,
                clean_test_loader,
                cfg.target_label,
                cfg.trigger_size,
                cfg.trigger_value,
                cfg.device,
            )

            print(f"[Round {round_id:02d}] MA={ma:.2f}% | ASR={asr:.2f}% | BA={ba:.2f}%")

            with open(metrics_path, "a", newline="") as f:
                writer = csv.writer(f)
                writer.writerow([round_id, f"{ma:.4f}", f"{asr:.4f}", f"{ba:.4f}"])

            torch.save(
                {
                    "round": round_id,
                    "head": current_head_state,
                    "backbone": current_backbone_state,
                    "tail": current_tail_state,
                    "ma": ma,
                    "asr": asr,
                    "ba": ba,
                    "config": vars(cfg),
                },
                os.path.join(cfg.save_dir, f"checkpoint_round_{round_id:02d}.pt"),
            )

    load_state_dict_to_module(head, current_head_state, cfg.device)
    load_state_dict_to_module(backbone, current_backbone_state, cfg.device)
    load_state_dict_to_module(tail, current_tail_state, cfg.device)

    final_ma = evaluate_clean(head, backbone, tail, clean_test_loader, cfg.device)
    final_asr = evaluate_asr(
        head,
        backbone,
        tail,
        clean_test_loader,
        cfg.target_label,
        cfg.trigger_size,
        cfg.trigger_value,
        cfg.device,
    )
    final_ba = evaluate_ba(
        head,
        backbone,
        tail,
        clean_test_loader,
        cfg.target_label,
        cfg.trigger_size,
        cfg.trigger_value,
        cfg.device,
    )

    final_path = os.path.join(cfg.save_dir, "final_results.txt")
    with open(final_path, "w") as f:
        f.write(f"Main Task Accuracy (MA): {final_ma:.4f}\n")
        f.write(f"Attack Success Rate (ASR): {final_asr:.4f}\n")
        f.write(f"Backdoor Accuracy (BA): {final_ba:.4f}\n")
        f.write(f"Config: {vars(cfg)}\n")
        f.write("Notes: ASR excludes original target-label samples; BA evaluates the entire triggered test set. target-label training samples are excluded from poisoning.\n")
        f.write("Paper target used here: CIFAR-10 Pixel Trigger, Table II direction: No Defense BA≈100.0/MA≈66.6; SafeSplit BA≈0.3/MA≈66.4.\n")
        f.write("Paper-aligned knobs: clients=10, malicious_clients=2, iid_rate=0.8, PDR/poison_fraction=0.75, red 4x4 bottom-right pixel trigger, target_label=2, augmentation ON.\n")

    print("\n========== Final ==========")
    print(f"Main Task Accuracy (MA): {final_ma:.2f}%")
    print(f"Attack Success Rate (ASR): {final_asr:.2f}%")
    print(f"Backdoor Accuracy (BA): {final_ba:.2f}%")
    print(f"Saved metrics to: {metrics_path}")
    print(f"Saved final results to: {final_path}")


def parse_args() -> Config:
    parser = argparse.ArgumentParser(description="SafeSplit-style split learning backdoor experiment on CIFAR-10")

    parser.add_argument("--data_root", type=str, default="./data")
    parser.add_argument("--batch_size", type=int, default=128)
    parser.add_argument("--num_workers", type=int, default=2)
    parser.add_argument("--rounds", type=int, default=50)
    parser.add_argument("--clients", type=int, default=10)
    parser.add_argument("--malicious_clients", type=int, default=2)
    parser.add_argument("--malicious_client_ids", type=str, default="8,9")
    parser.add_argument("--local_epochs", type=int, default=1)
    parser.add_argument("--malicious_local_epochs", type=int, default=2)
    parser.add_argument("--lr", type=float, default=0.001)
    parser.add_argument("--momentum", type=float, default=0.9)
    parser.add_argument("--weight_decay", type=float, default=1e-4)
    parser.add_argument("--seed", type=int, default=0)
    parser.add_argument("--iid_rate", type=float, default=0.8)
    parser.add_argument("--poison_fraction", type=float, default=0.75)
    parser.add_argument("--no_train_augmentation", action="store_true", help="Disable CIFAR-10 RandomCrop/RandomHorizontalFlip. Default: augmentation ON for higher MA.")
    parser.add_argument("--target_label", type=int, default=2)
    parser.add_argument("--trigger_size", type=int, default=4)
    parser.add_argument("--trigger_value", type=float, default=1.0)
    parser.add_argument("--no_defense", action="store_true", help="Disable SafeSplit defense. Default: defense ON.")
    parser.add_argument("--no_rotation", action="store_true", help="Disable rotational analysis. Default: rotation ON.")
    parser.add_argument("--low_freq_keep_ratio", type=float, default=0.02)
    parser.add_argument("--min_keep_per_tensor", type=int, default=2)
    parser.add_argument("--max_keep_per_tensor", type=int, default=8)
    parser.add_argument("--device", type=str, default="cuda" if torch.cuda.is_available() else "cpu")
    parser.add_argument("--save_dir", type=str, default="./outputs_paper_aligned")

    args, unknown = parser.parse_known_args()
    cfg = Config(
        data_root=args.data_root,
        batch_size=args.batch_size,
        num_workers=args.num_workers,
        rounds=args.rounds,
        clients=args.clients,
        malicious_clients=args.malicious_clients,
        malicious_client_ids=args.malicious_client_ids,
        local_epochs=args.local_epochs,
        malicious_local_epochs=args.malicious_local_epochs,
        lr=args.lr,
        momentum=args.momentum,
        weight_decay=args.weight_decay,
        seed=args.seed,
        iid_rate=args.iid_rate,
        poison_fraction=args.poison_fraction,
        train_augmentation=not args.no_train_augmentation,
        target_label=args.target_label,
        trigger_size=args.trigger_size,
        trigger_value=args.trigger_value,
        defense=not args.no_defense,
        use_rotation=not args.no_rotation,
        low_freq_keep_ratio=args.low_freq_keep_ratio,
        min_keep_per_tensor=args.min_keep_per_tensor,
        max_keep_per_tensor=args.max_keep_per_tensor,
        device=args.device,
        save_dir=args.save_dir,
    )
    return cfg


if __name__ == "__main__":
    cfg = parse_args()
    print(f"Initial Config: {cfg}")
    print("Hint: paper-aligned default = CIFAR-10 pixel trigger, PDR=0.75, iid_rate=0.8, clients=10, malicious=2, SafeSplit defense ON, rotation ON, augmentation ON. Use --no_defense for baseline.")
    run_experiment(cfg)


Initial Config: Config(data_root='./data', batch_size=128, num_workers=2, rounds=50, clients=10, malicious_clients=2, malicious_client_ids='8,9', local_epochs=1, malicious_local_epochs=2, lr=0.001, momentum=0.9, weight_decay=0.0001, seed=0, iid_rate=0.8, poison_fraction=0.75, train_augmentation=True, target_label=2, trigger_size=4, trigger_value=1.0, defense=True, use_rotation=True, low_freq_keep_ratio=0.02, min_keep_per_tensor=2, max_keep_per_tensor=8, device='cuda', save_dir='./outputs_paper_aligned')
Hint: paper-aligned default = CIFAR-10 pixel trigger, PDR=0.75, iid_rate=0.8, clients=10, malicious=2, SafeSplit defense ON, rotation ON, augmentation ON. Use --no_defense for baseline.

========== Split Parameter Check ==========
head params     : 9,536
backbone params : 11,166,976
tail params     : 5,130
total params    : 11,181,642
Step 001/500 | client=00 | benign    | epochs=1 | loss=2.1288 | chosen_checkpoint=1 | accepted
Step 002/500 | client=01 | benign    | epochs=1 | loss=1.99